In [ ]:
from google.colab import drive
drive.mount('/content/gdrive/')

Drive already mounted at /content/gdrive/; to attempt to forcibly remount, call drive.mount("/content/gdrive/", force_remount=True).


In [ ]:
!pip install --upgrade tensorflow


In [ ]:
import os
import tensorflow as tf
import datetime
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import scipy as sp
import cv2
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.models import Sequential, Model, load_model
from tensorflow.keras.layers import Input, Dense, Conv2D, MaxPool2D, BatchNormalization,  GlobalAveragePooling2D
from tensorflow.keras.callbacks import TensorBoard
import warnings
warnings.filterwarnings("ignore")
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

In [ ]:
%cd /content/gdrive/MyDrive/archive

/content/gdrive/MyDrive/archive


In [ ]:
train_path = "/content/gdrive/MyDrive/archive/train"
valid_path = "/content/gdrive/MyDrive/archive/valid"
test_path = "/content/gdrive/MyDrive/archive/test"

In [ ]:
import tensorflow as tf
im_size = 256
image_resize = (im_size, im_size, 3)
batch_size_training = 100
batch_size_validation = 100
batch_size_test = 100
num_classes = 2

In [ ]:
data_generator = ImageDataGenerator(dtype='float32', rescale= 1./255.)
train_generator = data_generator.flow_from_directory(train_path,
                                                   batch_size = batch_size_training,
                                                   target_size = (im_size, im_size),
                                                   class_mode = 'categorical')

valid_generator = data_generator.flow_from_directory(valid_path,
                                                   batch_size = batch_size_validation,
                                                   target_size = (im_size, im_size),
                                                   class_mode = 'categorical')
class_mapping = train_generator.class_indices
class_mapping
first_batch_train = next(train_generator)
first_batch_valid = next(valid_generator)
first_batch_train
first_batch_valid

Found 30259 images belonging to 2 classes.
Found 6301 images belonging to 2 classes.


(array([[[[0.25882354, 0.36862746, 0.21960786],
          [0.2784314 , 0.37647063, 0.22352943],
          [0.2784314 , 0.36078432, 0.20784315],
          ...,
          [0.16862746, 0.3019608 , 0.20392159],
          [0.16470589, 0.29803923, 0.20000002],
          [0.18039216, 0.3137255 , 0.21568629]],
 
         [[0.28235295, 0.38431376, 0.23137257],
          [0.27058825, 0.35686275, 0.20784315],
          [0.30588236, 0.37647063, 0.227451  ],
          ...,
          [0.17254902, 0.3137255 , 0.21176472],
          [0.16078432, 0.3019608 , 0.20000002],
          [0.18431373, 0.3254902 , 0.22352943]],
 
         [[0.2784314 , 0.3803922 , 0.227451  ],
          [0.28235295, 0.36862746, 0.21568629],
          [0.32941177, 0.40000004, 0.24313727],
          ...,
          [0.15294118, 0.29411766, 0.19215688],
          [0.16078432, 0.3019608 , 0.20000002],
          [0.16862746, 0.30980393, 0.20784315]],
 
         ...,
 
         [[0.20000002, 0.30980393, 0.21176472],
          [0.22745

In [ ]:

model = load_model('/content/gdrive/MyDrive/archive/saved_model/custom_best_model.h5')

In [ ]:

def base_model(input_shape, repetitions):
  input_ = tf.keras.layers.Input(shape=input_shape, name='input')
  x = input_
  for i in range(repetitions):
    n_filters = 2**(4 + i)
    x = Conv2D(n_filters, 3, activation='relu')(x)
    x = BatchNormalization()(x)
    x = MaxPool2D(2)(x)

  return x, input_

def final_model(input_shape, repetitions):

    x, input_ = base_model(input_shape, repetitions)

    x = Conv2D(64, 3, activation='relu')(x)
    x = GlobalAveragePooling2D()(x)
    class_out = Dense(num_classes, activation='softmax', name='class_out')(x)

    model = Model(inputs=input_, outputs=class_out)

    print(model.summary())
    return model

In [ ]:
model = final_model(image_resize, 4)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)                   │ (None, 256, 256, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d (Conv2D)                      │ (None, 254, 254, 16)        │             448 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 254, 254, 16)        │              64 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 127, 127, 16)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 125, 125, 32)        │           4,640 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 125, 125, 32)        │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 62, 62, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 60, 60, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 60, 60, 64)          │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 30, 30, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 28, 28, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_3                │ (None, 28, 28, 128)         │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_3 (MaxPooling2D)       │ (None, 14, 14, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_4 (Conv2D)                    │ (None, 12, 12, 64)          │          73,792 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 64)                  │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ class_out (Dense)                    │ (None, 2)                   │             130 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 172,322 (673.13 KB)

 Trainable params: 171,842 (671.26 KB)

 Non-trainable params: 480 (1.88 KB)

None


In [ ]:
get_ipython().system('rm -rf logs')

In [ ]:
model.compile(
    optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
)

In [ ]:
logdir = os.path.join("logs", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
tensorboard_callback = tf.keras.callbacks.TensorBoard(logdir)
checkpoint = tf.keras.callbacks.ModelCheckpoint('saved_model/custom_best_model.keras', monitor='val_accuracy', verbose=1, save_best_only=True, mode='max')


In [ ]:
callbacks_list = [checkpoint, tensorboard_callback]

In [ ]:
num_epochs = 1
steps_per_epoch_training = len(train_generator)
steps_per_epoch_validation = len(valid_generator)

In [ ]:
history = model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch_training,
    epochs=num_epochs,
    validation_data=valid_generator,
    validation_steps=steps_per_epoch_validation,
    verbose=1,
    callbacks=callbacks_list,
)


303/303 ━━━━━━━━━━━━━━━━━━━━ 0s 35s/step - accuracy: 0.9141 - loss: 0.2360 
Epoch 1: val_accuracy improved from -inf to 0.56356, saving model to saved_model/custom_best_model.keras
303/303 ━━━━━━━━━━━━━━━━━━━━ 12171s 39s/step - accuracy: 0.9141 - loss: 0.2358 - val_accuracy: 0.5636 - val_loss: 1.0854


In [ ]:
from tensorflow.keras.layers import Dropout
from tensorflow.keras.callbacks import ReduceLROnPlateau
def base_model1(input_shape, repetitions):
    input_ = tf.keras.layers.Input(shape=input_shape, name='input')
    x = input_

    for i in range(repetitions):
        n_filters = 2**(4 + i)
        x = Conv2D(n_filters, 3, activation='relu', padding='same')(x)
        x = BatchNormalization()(x)
        x = Conv2D(n_filters, 3, activation='relu', padding='same')(x)
        x = BatchNormalization()(x)
        x = MaxPool2D(2)(x)
        x = Dropout(0.25)(x)

    return x, input_

def final_model(input_shape, repetitions):

    x, input_ = base_model1(input_shape, repetitions)

    x = Conv2D(64, 3, activation='relu')(x)
    x = GlobalAveragePooling2D()(x)
    class_out = Dense(num_classes, activation='softmax', name='class_out')(x)

    model = Model(inputs=input_, outputs=class_out)

    print(model.summary())
    return model

reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6)

history = model.fit_generator(
    train_generator,
    steps_per_epoch=steps_per_epoch_training,
    epochs=num_epochs,
    validation_data=valid_generator,
    validation_steps=steps_per_epoch_validation,
    verbose=1,
    callbacks=[callbacks_list, reduce_lr],
)
model1 = final_model(image_resize, 4)
get_ipython().system('rm -rf logs')
model1.compile(
    optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
)
logdir = os.path.join("logs", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
tensorboard_callback1 = tf.keras.callbacks.TensorBoard(logdir)
checkpoint1 = tf.keras.callbacks.ModelCheckpoint('saved_model/custom_best_model1.h5', monitor='val_accuracy', verbose=1, save_best_only=True, mode='max')
callbacks_list1 = [checkpoint1, tensorboard_callback1]
history = model1.fit_generator(
    train_generator,
    steps_per_epoch=steps_per_epoch_training,
    epochs=num_epochs,
    validation_data=valid_generator,
    validation_steps=steps_per_epoch_validation,
    verbose=1,
    callbacks=[callbacks_list1],
)

AttributeError: 'Functional' object has no attribute 'fit_generator'

In [ ]:
test_generator = data_generator.flow_from_directory(test_path,
                                                   batch_size = batch_size_test,
                                                   target_size = (im_size, im_size),
                                                   class_mode = 'categorical')

In [ ]:
filenames = test_generator.filenames
pred = model.predict(test_generator, steps=len(test_generator), verbose=1).round(3)


In [ ]:
filenames_df = pd.DataFrame(filenames, columns=['File Path'])
pred_df = pd.DataFrame(pred, columns=['No Wildfire Probability', 'Wildfire Probability'])
model_predictions = pd.concat([filenames_df, pred_df], axis=1)
model_predictions
file_name='/content/gdrive/MyDrive/archive/predictions/custom_model_predictions.csv'
model_predictions.to_csv(file_name, sep=',', encoding='utf-8')

In [ ]:
cam_model  = Model(inputs=model.input, outputs=(model.layers[-3].output,model.layers[-1].output))
cam_model.summary()

In [ ]:
outputs = [layer.output for layer in model.layers[1:9]]
vis_model = Model(model.input, outputs)
layer_names = []
for layer in outputs:
    layer_names.append(layer.name.split("/")[0])
gap_weights = model.layers[-1].get_weights()[0]
gap_weights.shape


In [ ]:
cam_model.save('saved_model/cam_model')
print("Model saved!")

In [ ]:
import firebase_admin
from firebase_admin import credentials, messaging
cred = credentials.Certificate('/content/gdrive/MyDrive/firebase.json')
firebase_admin.initialize_app(cred)




In [ ]:
!pip install --upgrade firebase-admin

In [ ]:
import requests
import cv2
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from geopy.geocoders import Nominatim
from IPython.display import HTML
import firebase_admin
from firebase_admin import credentials, messaging
api_key = 'b9c155a04ae448dd8e8bffabd03dd100'
im_size = 256

    #messaging.send(messaging.Message(notification=notification, token="d3JQfW6VSUuHI0VMTLLJx-:APA91bFIYv3PIndkfOwJcms1qpXjpehKySwZ5Z5MrmlurqLTdaTMJKQv8EUH2U8qX64Wg-I_LN8XwEPpb9Pj9b-LK3UeauJjdBO46N9SmP9o14dfLp7hi8Yb9LrjpArZVejYiD-imo_W"))
def send_notification(message, latitude, longitude):
    maps_link = f"https://www.google.com/maps/search/?api=1&query={latitude},{longitude}"
    message_with_link = f"{message}\nLocation: {maps_link}"
    notification = messaging.Notification(title="Fire Detection", body=message_with_link)
    messaging.send(messaging.Message(notification=notification, token="f-x3jIMvRdWlRdOcJp4Az4:APA91bGRs0U-Kz--pR4unLh4662FSLl68zAuRIqp4rasQgEgrVq5THM-ffQZBokwkwBpjwK8_LnwSsVwfvBYQmphXTKzXLgNpNw4e8ZW8pjbgyf7-8DENzqc6R2ZfZx0RpVrGnx1kRhu"))
def reverse_geocode_opencage(latitude, longitude):
    url = f'https://api.opencagedata.com/geocode/v1/json?q={latitude}+{longitude}&key={api_key}'
    try:
        response = requests.get(url)
        response.raise_for_status()  # Raise an error for bad responses (4xx or 5xx)
        data = response.json()
        if data['results']:
            return data['results'][0]['formatted']
        else:
            return 'No results found'
    except requests.exceptions.RequestException as e:
        print(f"Error retrieving reverse geocoding data: {e}")
        return f'Error: {e}'


def convert_and_classify(image):
    global gap_weights, im_size, cam_model
    filename = image.split('/')[-1]
    coords = filename.split('.jpg')[0]
    latitude, longitude = coords.split(',')
    longitude = float(longitude)
    latitude = float(latitude)

    formatted_address = reverse_geocode_opencage(latitude, longitude)
    print(f"Formatted Address: {formatted_address}")

    maps_link = f"https://www.google.com/maps/search/?api=1&query={latitude},{longitude}"
    display(HTML(f'<a href="{maps_link}" target="_blank">Google Maps Link</a>'))
    img = cv2.imread(image)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (im_size, im_size)) / 255.0
    tensor_image = np.expand_dims(img, axis=0)
    features, results = cam_model.predict(tensor_image)
    fire_probability = results[0][1]
    fire_probability1=results[0][0]
    if fire_probability < fire_probability1:
        fire_label = "Fire"
        send_notification("Fire detected in satellite image",latitude,longitude)
    else:
        fire_label = "No Fire"
    gap_weights = cam_model.layers[-1].get_weights()[0]
    show_cam(tensor_image, features, results, gap_weights, im_size, fire_label)
def show_cam(image_value, features, results, gap_weights, im_size, fire_label):
    features_for_img = features[0]
    class_activation_weights = gap_weights[:, 0]
    class_activation_features = sp.ndimage.zoom(features_for_img, (im_size / features_for_img.shape[0], im_size / features_for_img.shape[1], 1), order=2)
    cam_output = np.dot(class_activation_features, class_activation_weights)

    plt.figure(figsize=(6,6))
    plt.imshow(cam_output, cmap='jet', alpha=0.5)
    plt.imshow(tf.squeeze(image_value), alpha=0.5)
    plt.figtext(.5, .05, f"Predicted: {fire_label}", ha="center", fontsize=12,
                bbox={"facecolor": "green" if fire_label == "No Fire" else "red", "alpha": 0.5, "pad": 3})
    plt.colorbar()
    plt.show()
convert_and_classify('/content/gdrive/MyDrive/archive/test/wildfire/-59.03238,51.85132.jpg')


In [ ]:
convert_and_classify('/content/gdrive/MyDrive/archive/test/wildfire/-59.03238,51.85132.jpg')

In [ ]:
from google.colab import files
from keras.preprocessing import image
from numpy import asarray

In [ ]:
convert_and_classify('/content/gdrive/MyDrive/archive/test/wildfire/-65.11769,48.622.jpg')

In [ ]:
uploaded = files.upload()

In [ ]:
for fn in uploaded.keys():
  width = im_size
  height = im_size
  dim = (width, height)
  path = '/content/gdrive/MyDrive/archive/' + fn
  img = cv2.imread(path)
  convert_and_classify(path)